<a href="https://colab.research.google.com/github/ibtihal7alharbi-tech/Computer-vision-project-/blob/main/Face_nose_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Specialized Detection Sub-Systems
Before the final decision-making process, the Mahd architecture relies on two distinct, high-precision models that analyze different anatomical hierarchies of the infant.

# Respiratory Pathway & Facial Feature Analytics
The primary objective of this model is to serve as a Life-Sign Guard. While a baby might be in a certain physical position, the most critical safety factor is whether the breathing pathways (nose) are unobstructed.

Granular Detection: This model is trained to identify micro-features such as the nasal tip, the bridge of the nose, and the oral region.

Obstruction Logic: If the model fails to detect these features with a high confidence score, it flags a potential "Occlusion Event." This is vital because SIDS risks increase significantly when a baby’s face is pressed against bedding, even if the body orientation appears normal.

In [ ]:
!pip install roboflow ultralytics -q

!pip install roboflow

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="ZeJKDEWv7ZknZBJqtk1s")
project = rf.workspace("ibtihals-workspace").project("baby-project-s4cck")
version = project.version(2)
dataset = version.download("yolov8-obb")

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=20,
    imgsz=640,
    plots=True,
    device=0
)


In [ ]:
import shutil
import os

metrics = model.val()
print("\n" + "="*30)
print("📈 Final Performance Metrics:")
print(f"Precision: {metrics.results_dict['metrics/precision(B)']:.4f}")
print(f"Recall:    {metrics.results_dict['metrics/recall(B)']:.4f}")
print(f"mAP50:     {metrics.results_dict['metrics/mAP50(B)']:.4f}")
print("="*30)

best_model_path = 'runs/detect/train/weights/best.pt'
target_path = 'BabyGuard_Detection_v1.pt'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, target_path)
    print(f" Model saved successfully as: {target_path}")
else:
    print(" Warning")

test_load = YOLO(target_path)
print(" Model is ready for integration!")

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

best_model = YOLO('/content/runs/detect/train/weights/best.pt')

results = best_model.predict(source='//content/baby-project-2/test/images/baby-sleep44_jpg.rf.e59fbb8ffbe39aedeb446026f4baf7fb.jpg', conf=0.4)

res_plotted = results[0].plot()
cv2_imshow(res_plotted)